# Notebook 15 — Prototype Update and Recovery

**Repo:** `int_serialization_benchmark-rml`  
**Layer:** `rml_extension/notebooks/`

Notebook 14 detected online drift when existing mixed-regime prototypes stopped explaining the stream.

Notebook 15 closes the recovery loop:

- collect drift windows,
- estimate a new candidate prototype,
- add it to the prototype bank,
- rerun decomposition,
- test whether residuals and drift alarms decrease.

Constraint view:
> drift detection becomes useful when the runtime can recover by updating its prototype memory.

## Goals

1. Load Notebook 14 drift-detection results when available.
2. Load Notebook 13 mixed-regime decomposition results when available.
3. Identify drift / unknown candidate windows.
4. Build a candidate prototype from drift windows.
5. Compare old vs updated prototype-bank reconstruction.
6. Estimate recovery:
   - residual reduction
   - alarm reduction
   - unknown-candidate reduction
   - policy-stability improvement
7. Export CSV, JSON, Markdown report, and PNG figures.

In [ ]:
from pathlib import Path
import json
import zipfile
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
from scipy.optimize import nnls

cwd = Path.cwd()
candidates = [
    cwd,
    cwd.parent,
    cwd.parent.parent,
    Path("/content/int_serialization_benchmark-rml"),
    Path("/content"),
]

REPO_ROOT = None
for c in candidates:
    if (c / "rml_extension").exists() or (c / "configs").exists():
        REPO_ROOT = c
        break

if REPO_ROOT is None:
    REPO_ROOT = cwd

RML_ROOT = REPO_ROOT / "rml_extension" if (REPO_ROOT / "rml_extension").exists() else REPO_ROOT

RESULTS_DIR = RML_ROOT / "results"
FIGURES_DIR = RML_ROOT / "figures"
REPORTS_DIR = RML_ROOT / "reports"

for d in [RESULTS_DIR, FIGURES_DIR, REPORTS_DIR]:
    d.mkdir(parents=True, exist_ok=True)

print("REPO_ROOT:", REPO_ROOT)
print("RML_ROOT:", RML_ROOT)

## Load drift and mixture tables

Uses Notebook 14 and Notebook 13 outputs if available. Otherwise creates a synthetic recovery example.

In [ ]:
drift_path = RESULTS_DIR / "notebook14_online_drift_detection.csv"
mix_path = RESULTS_DIR / "notebook13_mixed_regime_decomposition.csv"

if drift_path.exists():
    drift = pd.read_csv(drift_path)
    print("Loaded:", drift_path)
else:
    drift = None

if mix_path.exists():
    mix = pd.read_csv(mix_path)
    print("Loaded:", mix_path)
else:
    mix = None

if drift is None or mix is None:
    print("Missing prior outputs; creating fallback recovery table.")
    rng = np.random.default_rng(42)
    n = 180
    rows = []
    for i in range(n):
        # Baseline features.
        entropy_norm = rng.uniform(0.1, 0.9)
        repetition = rng.uniform(0.0, 1.0)
        locality = rng.uniform(0.0, 1.0)
        reuse = rng.uniform(0.0, 1.0)
        branch = rng.uniform(0.0, 0.8)
        coherence = rng.uniform(0.1, 0.9)
        pressure = rng.uniform(0.1, 0.9)
        residual = abs(rng.normal(0.035, 0.015))
        entropy = abs(rng.normal(1.05, 0.25))
        alarm = False
        unknown = False

        # Inject unknown prototype interval.
        if 105 <= i <= 135:
            entropy_norm = rng.uniform(0.40, 0.65)
            repetition = rng.uniform(0.35, 0.55)
            locality = rng.uniform(0.35, 0.55)
            reuse = rng.uniform(0.30, 0.50)
            branch = rng.uniform(0.55, 0.75)
            coherence = rng.uniform(0.35, 0.55)
            pressure = rng.uniform(0.60, 0.80)
            residual = abs(rng.normal(0.13, 0.02))
            entropy = abs(rng.normal(1.75, 0.20))
            alarm = i in range(106, 109)
            unknown = True

        rows.append({
            "window_id": i,
            "entropy_norm": entropy_norm,
            "repetition_ratio": repetition,
            "locality_small_delta_ratio": locality,
            "cache_window_reuse_proxy": reuse,
            "branch_norm": branch,
            "coherence_score": coherence,
            "hardware_pressure_proxy": pressure,
            "reconstruction_residual": residual,
            "mixture_entropy": entropy,
            "drift_alarm": alarm,
            "unknown_regime_candidate": unknown,
            "mixture_policy": "hybrid",
        })

    merged = pd.DataFrame(rows)
else:
    # Merge useful columns from both if needed.
    merged = mix.copy()
    if "window_id" in drift.columns:
        drift_cols = [
            c for c in [
                "window_id",
                "drift_score",
                "drift_warning",
                "drift_alarm",
                "unknown_regime_candidate",
                "residual_z",
                "entropy_z",
                "policy_switch_rate",
                "dominant_switch_rate",
            ] if c in drift.columns
        ]
        merged = merged.merge(drift[drift_cols], on="window_id", how="left", suffixes=("", "_drift"))

merged.head()

## Define original prototype bank

Same feature basis as Notebook 13.

In [ ]:
prototype_rows = [
    {
        "regime": "low_entropy_repeating",
        "entropy_norm": 0.10,
        "repetition_ratio": 0.98,
        "locality_small_delta_ratio": 1.00,
        "cache_window_reuse_proxy": 0.94,
        "branch_norm": 0.02,
        "coherence_score": 0.95,
        "hardware_pressure_proxy": 0.02,
    },
    {
        "regime": "sequential_ids",
        "entropy_norm": 0.90,
        "repetition_ratio": 0.00,
        "locality_small_delta_ratio": 1.00,
        "cache_window_reuse_proxy": 0.00,
        "branch_norm": 0.20,
        "coherence_score": 0.55,
        "hardware_pressure_proxy": 0.25,
    },
    {
        "regime": "uniform_32bit",
        "entropy_norm": 1.00,
        "repetition_ratio": 0.00,
        "locality_small_delta_ratio": 0.00,
        "cache_window_reuse_proxy": 0.00,
        "branch_norm": 0.72,
        "coherence_score": 0.08,
        "hardware_pressure_proxy": 0.88,
    },
    {
        "regime": "zipfian_smallints",
        "entropy_norm": 0.18,
        "repetition_ratio": 0.80,
        "locality_small_delta_ratio": 0.27,
        "cache_window_reuse_proxy": 0.35,
        "branch_norm": 0.72,
        "coherence_score": 0.42,
        "hardware_pressure_proxy": 0.72,
    },
    {
        "regime": "clustered_ranges",
        "entropy_norm": 0.55,
        "repetition_ratio": 0.98,
        "locality_small_delta_ratio": 0.00,
        "cache_window_reuse_proxy": 0.01,
        "branch_norm": 0.79,
        "coherence_score": 0.18,
        "hardware_pressure_proxy": 0.98,
    },
]

feature_cols = [
    "entropy_norm",
    "repetition_ratio",
    "locality_small_delta_ratio",
    "cache_window_reuse_proxy",
    "branch_norm",
    "coherence_score",
    "hardware_pressure_proxy",
]

prototypes_old = pd.DataFrame(prototype_rows)

# Ensure feature columns exist in merged.
for c in feature_cols:
    if c not in merged.columns:
        # Try reconstructed fallback if coming from Notebook 13.
        rc = f"reconstructed_{c}"
        if rc in merged.columns:
            merged[c] = merged[rc]
        else:
            merged[c] = 0.5

prototypes_old

## Identify drift windows and learn candidate prototype

Candidate prototype is the mean feature vector over unknown/drift windows.

If no drift windows exist, the highest-residual windows are used.

In [ ]:
work = merged.copy().sort_values("window_id").reset_index(drop=True)

for flag in ["drift_alarm", "unknown_regime_candidate"]:
    if flag not in work.columns:
        work[flag] = False
    work[flag] = work[flag].fillna(False).astype(bool)

if "reconstruction_residual" not in work.columns:
    work["reconstruction_residual"] = 0.0

drift_mask = work["unknown_regime_candidate"] | work["drift_alarm"]

if drift_mask.sum() < 5:
    cutoff = work["reconstruction_residual"].quantile(0.90)
    drift_mask = work["reconstruction_residual"] >= cutoff

drift_windows = work[drift_mask].copy()

candidate_vec = drift_windows[feature_cols].mean().clip(0, 1)

new_proto = {"regime": "learned_drift_prototype"}
new_proto.update(candidate_vec.to_dict())

prototypes_new = pd.concat([prototypes_old, pd.DataFrame([new_proto])], ignore_index=True)

print("Drift windows used:", len(drift_windows))
pd.DataFrame([new_proto])

## Decompose with old and updated prototype banks

In [ ]:
def decompose_with_bank(df, prototypes, label):
    regimes = list(prototypes["regime"])
    A = prototypes[feature_cols].to_numpy().T

    rows = []
    for _, row in df.iterrows():
        b = row[feature_cols].to_numpy(float)
        w, residual = nnls(A, b)
        if w.sum() > 0:
            w = w / w.sum()
        recon = A @ w

        out = {
            "window_id": int(row["window_id"]),
            f"{label}_residual": float(np.linalg.norm(b - recon)),
            f"{label}_dominant_prototype": regimes[int(np.argmax(w))],
            f"{label}_mixture_entropy": float(-(w[w > 0] * np.log2(w[w > 0])).sum()) if np.any(w > 0) else 0.0,
        }
        for regime, weight in zip(regimes, w):
            out[f"{label}_weight_{regime}"] = float(weight)
        rows.append(out)

    return pd.DataFrame(rows)

old_dec = decompose_with_bank(work, prototypes_old, "old")
new_dec = decompose_with_bank(work, prototypes_new, "new")

recovery = work.merge(old_dec, on="window_id").merge(new_dec, on="window_id")
recovery["residual_reduction"] = recovery["old_residual"] - recovery["new_residual"]
recovery["residual_reduction_pct"] = 100.0 * recovery["residual_reduction"] / recovery["old_residual"].replace(0, np.nan)
recovery["residual_reduction_pct"] = recovery["residual_reduction_pct"].fillna(0.0)

recovery[["window_id", "old_residual", "new_residual", "residual_reduction", "new_dominant_prototype"]].head()

## Recompute drift alarms after prototype update

This tests whether adding the new prototype reduces mismatch.

In [ ]:
def rolling_z(series, window=15):
    s = pd.Series(series).astype(float)
    mu = s.shift(1).rolling(window, min_periods=5).mean().fillna(s.expanding().mean())
    sd = s.shift(1).rolling(window, min_periods=5).std()
    sd = sd.replace(0, np.nan).fillna(s.std() + 1e-9)
    return (s - mu) / (sd + 1e-9)

recovery["old_residual_z"] = rolling_z(recovery["old_residual"])
recovery["new_residual_z"] = rolling_z(recovery["new_residual"])
recovery["new_entropy_z"] = rolling_z(recovery["new_mixture_entropy"])

def positive_clip(s, cap=4.0):
    return np.clip(pd.Series(s).astype(float), 0, cap) / cap

# Use original drift if present, but recompute old/new comparable scores.
recovery["old_drift_score_recomputed"] = (
    0.55 * positive_clip(recovery["old_residual_z"]) +
    0.25 * positive_clip(rolling_z(recovery.get("mixture_entropy", recovery["old_mixture_entropy"]))) +
    0.20 * recovery.get("policy_switch_rate", pd.Series(0, index=recovery.index)).fillna(0)
).clip(0, 1)

recovery["new_drift_score"] = (
    0.55 * positive_clip(recovery["new_residual_z"]) +
    0.25 * positive_clip(recovery["new_entropy_z"]) +
    0.20 * recovery.get("policy_switch_rate", pd.Series(0, index=recovery.index)).fillna(0)
).clip(0, 1)

threshold = 0.55
recovery["old_alarm_recomputed"] = recovery["old_drift_score_recomputed"] >= threshold
recovery["new_alarm"] = recovery["new_drift_score"] >= threshold
recovery["recovered_alarm"] = recovery["old_alarm_recomputed"] & (~recovery["new_alarm"])

recovery[["window_id", "old_drift_score_recomputed", "new_drift_score", "old_alarm_recomputed", "new_alarm", "recovered_alarm"]].head()

## Prototype-aware policy update

Windows dominated by the new prototype are assigned `prototype_recovery`.
This marks areas that need either a new execution route or human inspection.

In [ ]:
def recovery_policy(row):
    if row["new_dominant_prototype"] == "learned_drift_prototype":
        return "prototype_recovery"
    dominant = row["new_dominant_prototype"]
    return {
        "low_entropy_repeating": "coherent_local",
        "sequential_ids": "hybrid",
        "uniform_32bit": "simd",
        "zipfian_smallints": "hybrid",
        "clustered_ranges": "guarded_fallback",
    }.get(dominant, "hybrid")

recovery["updated_policy"] = recovery.apply(recovery_policy, axis=1)

if "mixture_policy" in recovery.columns:
    recovery["policy_changed_after_update"] = recovery["updated_policy"] != recovery["mixture_policy"]
else:
    recovery["policy_changed_after_update"] = False

recovery[["window_id", "new_dominant_prototype", "updated_policy", "policy_changed_after_update"]].head()

## Export recovery tables

In [ ]:
csv_path = RESULTS_DIR / "notebook15_prototype_update_and_recovery.csv"
json_path = RESULTS_DIR / "notebook15_prototype_update_and_recovery.json"
proto_csv_path = RESULTS_DIR / "notebook15_updated_prototypes.csv"

recovery.to_csv(csv_path, index=False)
recovery.to_json(json_path, orient="records", indent=2)
prototypes_new.to_csv(proto_csv_path, index=False)

print("Saved:", csv_path)
print("Saved:", json_path)
print("Saved:", proto_csv_path)

## Figure 1 — Old vs new reconstruction residuals

In [ ]:
fig_path_1 = FIGURES_DIR / "notebook15_old_vs_new_residuals.png"

plt.figure(figsize=(12, 4))
plt.plot(recovery["window_id"], recovery["old_residual"], label="old prototype bank")
plt.plot(recovery["window_id"], recovery["new_residual"], label="updated prototype bank")
plt.xlabel("Window")
plt.ylabel("Reconstruction residual")
plt.title("Prototype Update: Old vs New Reconstruction Residuals")
plt.legend()
plt.tight_layout()
plt.savefig(fig_path_1, dpi=160)
plt.show()

print("Saved:", fig_path_1)

## Figure 2 — Residual reduction timeline

In [ ]:
fig_path_2 = FIGURES_DIR / "notebook15_residual_reduction_timeline.png"

plt.figure(figsize=(12, 4))
plt.plot(recovery["window_id"], recovery["residual_reduction"])
plt.axhline(0, linestyle="--")
plt.xlabel("Window")
plt.ylabel("Residual reduction")
plt.title("Prototype Update: Residual Reduction")
plt.tight_layout()
plt.savefig(fig_path_2, dpi=160)
plt.show()

print("Saved:", fig_path_2)

## Figure 3 — Drift score before and after update

In [ ]:
fig_path_3 = FIGURES_DIR / "notebook15_drift_score_before_after.png"

plt.figure(figsize=(12, 4))
plt.plot(recovery["window_id"], recovery["old_drift_score_recomputed"], label="old drift score")
plt.plot(recovery["window_id"], recovery["new_drift_score"], label="new drift score")
plt.axhline(threshold, linestyle="--", label="alarm threshold")
plt.xlabel("Window")
plt.ylabel("Drift score")
plt.title("Prototype Update: Drift Score Before vs After")
plt.legend()
plt.tight_layout()
plt.savefig(fig_path_3, dpi=160)
plt.show()

print("Saved:", fig_path_3)

## Figure 4 — Learned prototype feature vector

In [ ]:
fig_path_4 = FIGURES_DIR / "notebook15_learned_prototype_features.png"

learned = prototypes_new[prototypes_new["regime"] == "learned_drift_prototype"].iloc[0]

plt.figure(figsize=(10, 5))
plt.bar(feature_cols, [learned[c] for c in feature_cols])
plt.xticks(rotation=45, ha="right")
plt.ylabel("Feature value")
plt.title("Learned Drift Prototype: Feature Vector")
plt.tight_layout()
plt.savefig(fig_path_4, dpi=160)
plt.show()

print("Saved:", fig_path_4)

## Figure 5 — Dominant prototype timeline after update

In [ ]:
fig_path_5 = FIGURES_DIR / "notebook15_updated_dominant_prototype_timeline.png"

labels = sorted(recovery["new_dominant_prototype"].unique())
lab_to_id = {lab: i for i, lab in enumerate(labels)}

plt.figure(figsize=(12, 4))
plt.step(recovery["window_id"], recovery["new_dominant_prototype"].map(lab_to_id), where="mid")
plt.yticks(list(lab_to_id.values()), list(lab_to_id.keys()))
plt.xlabel("Window")
plt.ylabel("Dominant prototype")
plt.title("Updated Prototype Bank: Dominant Prototype Timeline")
plt.tight_layout()
plt.savefig(fig_path_5, dpi=160)
plt.show()

print("Saved:", fig_path_5)

## Figure 6 — Alarm recovery summary

In [ ]:
fig_path_6 = FIGURES_DIR / "notebook15_alarm_recovery_summary.png"

counts = pd.DataFrame([
    {"category": "old alarms", "count": int(recovery["old_alarm_recomputed"].sum())},
    {"category": "new alarms", "count": int(recovery["new_alarm"].sum())},
    {"category": "recovered alarms", "count": int(recovery["recovered_alarm"].sum())},
    {"category": "new prototype dominant", "count": int((recovery["new_dominant_prototype"] == "learned_drift_prototype").sum())},
])

plt.figure(figsize=(8, 5))
plt.bar(counts["category"], counts["count"])
plt.xticks(rotation=30, ha="right")
plt.ylabel("Window count")
plt.title("Prototype Update: Alarm Recovery Summary")
plt.tight_layout()
plt.savefig(fig_path_6, dpi=160)
plt.show()

print("Saved:", fig_path_6)

## Lab-report summary

In [ ]:
report_path = REPORTS_DIR / "report_15_prototype_update_and_recovery.md"

drift_region = drift_mask.reset_index(drop=True)

summary = {
    "windows": int(len(recovery)),
    "drift_windows_used_for_prototype": int(len(drift_windows)),
    "mean_old_residual": float(recovery["old_residual"].mean()),
    "mean_new_residual": float(recovery["new_residual"].mean()),
    "mean_residual_reduction": float(recovery["residual_reduction"].mean()),
    "mean_residual_reduction_pct": float(recovery["residual_reduction_pct"].mean()),
    "old_alarm_count": int(recovery["old_alarm_recomputed"].sum()),
    "new_alarm_count": int(recovery["new_alarm"].sum()),
    "recovered_alarm_count": int(recovery["recovered_alarm"].sum()),
    "new_prototype_dominant_windows": int((recovery["new_dominant_prototype"] == "learned_drift_prototype").sum()),
}

policy_counts = recovery["updated_policy"].value_counts().rename_axis("policy").reset_index(name="count")

lines = [
    "# Report 15 — Prototype Update and Recovery",
    "",
    "This report updates the prototype bank using drift windows and tests whether reconstruction and drift alarms improve.",
    "",
    "Constraint view:",
    "> drift detection becomes useful when the runtime can recover by updating its prototype memory.",
    "",
    "## Generated outputs",
    "",
    f"- Metrics CSV: `{csv_path}`",
    f"- Metrics JSON: `{json_path}`",
    f"- Updated prototypes CSV: `{proto_csv_path}`",
    f"- Figure: `{fig_path_1}`",
    f"- Figure: `{fig_path_2}`",
    f"- Figure: `{fig_path_3}`",
    f"- Figure: `{fig_path_4}`",
    f"- Figure: `{fig_path_5}`",
    f"- Figure: `{fig_path_6}`",
    "",
    "## Summary",
    "",
    pd.DataFrame([summary]).to_markdown(index=False),
    "",
    "## Learned drift prototype",
    "",
    pd.DataFrame([new_proto]).to_markdown(index=False),
    "",
    "## Updated policy counts",
    "",
    policy_counts.to_markdown(index=False),
    "",
    "## Interpretation",
    "",
    "- A new prototype is learned from drift / unknown-candidate windows.",
    "- Recovery is measured by comparing old and new reconstruction residuals.",
    "- Alarm reduction indicates whether the new prototype explains previously anomalous windows.",
    "- Windows dominated by the learned prototype become candidates for a new execution policy or deeper inspection.",
    "",
    "## Next step",
    "",
    "Notebook 16 can build a prototype memory bank with aging, pruning, and stability scores.",
]

report_path.write_text("\n".join(lines))
print("Saved:", report_path)

## Optional: download output bundle in Colab

Uncomment the following cell if you are running this notebook in Google Colab and want to download generated outputs.

In [ ]:
# OPTIONAL COLAB DOWNLOAD
#
# EXPORT_NAME = "notebook15_prototype_update_and_recovery_outputs.zip"
# export_path = RML_ROOT / EXPORT_NAME
#
# with zipfile.ZipFile(export_path, "w", zipfile.ZIP_DEFLATED) as zf:
#     for folder in [RESULTS_DIR, FIGURES_DIR, REPORTS_DIR]:
#         for p in folder.glob("notebook15_*"):
#             zf.write(p, arcname=str(p.relative_to(RML_ROOT)))
#         for p in folder.glob("report_15_*"):
#             zf.write(p, arcname=str(p.relative_to(RML_ROOT)))
#
# from google.colab import files
# files.download(str(export_path))